# NB00 — MicrobeAtlas Terrestrial Sample QC and Spatial Thinning

Fetch all MicrobeAtlas samples with valid lat/lon and terrestrial `environments` labels,
apply 0.45° spatial thinning (one sample per grid cell, ranked by data completeness),
and save the thinned sample list for NB01 (CWM construction).

**Key pitfall:** `Env_Level_1` / `Env_Level_2` are NOT columns — filter via `environments` (pipe-separated text).

**V-region** is OTU-level, not sample-level. Assignment deferred to NB01 after OTU fetch.

In [1]:
import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H
apply_style()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
except Exception as e:
    print(f'JupyterHub import failed ({e}), trying standalone')
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm')
DATA = PROJECT / 'data'
DATA.mkdir(exist_ok=True)
FIGS = PROJECT / 'figures'
FIGS.mkdir(exist_ok=True)
print(f'Spark {spark.version}')
print(f'DATA: {DATA}')
print(f'FIGS: {FIGS}')

Spark 4.0.1
DATA: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data
FIGS: /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/figures


In [2]:
# Count total and lat/lon-valid samples for attrition tracking.
# lat: FLOAT (388,579 non-null) — canonical coordinate column
# latitude_parsed: STRING (854,916 non-null when lat IS NULL) with actual numeric coords
# lat_field / lat_field_value: contain field NAME labels, not coordinates — do NOT use
# lat_field_value has malformed values ('51.115.861') — skip entirely

n_total = spark.sql("SELECT COUNT(*) AS n FROM arkinlab.microbeatlas.sample_metadata").collect()[0]['n']

n_latlon = spark.sql("""
    SELECT COUNT(*) AS n
    FROM arkinlab.microbeatlas.sample_metadata
    WHERE COALESCE(lat, CAST(latitude_parsed AS DOUBLE)) IS NOT NULL
      AND COALESCE(lon, CAST(longitude_parsed AS DOUBLE)) IS NOT NULL
""").collect()[0]['n']

print(f'Total samples:          {n_total:>10,}')
print(f'Valid lat/lon:          {n_latlon:>10,}')

Total samples:           1,884,129
Valid lat/lon:           1,241,411


In [3]:
# Fetch terrestrial samples with lat/lon and covariate columns.
# Use COALESCE(lat, CAST(latitude_parsed AS DOUBLE)) — both confirmed safe to cast directly.
# lat_field_value excluded: contains malformed values ('51.115.861') that break CAST.

INCLUDE = ['soil', 'terrestrial', 'forest', 'desert', 'shrub', 'peatland',
           'tundra', 'cave', 'mine', 'agricultural', 'grassland', 'rhizosphere',
           'litter', 'compost', 'permafrost']
EXCLUDE = ['marine', 'ocean', 'freshwater', 'host', 'gut', 'oral', 'skin',
           'feces', 'faeces', 'wastewater', 'sewage', 'lake', 'river',
           'aquatic', 'pond', 'drinking water', 'deep-sea', 'seawater',
           'sediment']

inc = ' OR '.join(f"LOWER(environments) LIKE '%{e}%'" for e in INCLUDE)
exc = ' OR '.join(f"LOWER(environments) LIKE '%{e}%'" for e in EXCLUDE)

query = f"""
SELECT
    sample_id,
    COALESCE(lat, CAST(latitude_parsed  AS DOUBLE)) AS lat,
    COALESCE(lon, CAST(longitude_parsed AS DOUBLE)) AS lon,
    environments,
    ph,
    olm_soil_ph_0cm_H2O,
    dem_elevation_m,
    era5_mean_2m_air_temp_k,
    era5_total_precipitation_mm,
    lights_radiance_nanow_cm2_sr,
    ndvi,
    run_acc
FROM arkinlab.microbeatlas.sample_metadata
WHERE
    COALESCE(lat, CAST(latitude_parsed  AS DOUBLE)) IS NOT NULL
    AND COALESCE(lon, CAST(longitude_parsed AS DOUBLE)) IS NOT NULL
    AND ({inc})
    AND NOT ({exc})
"""

df = spark.sql(query).toPandas()
df.attrs = {}
print(f'Terrestrial samples with lat/lon: {len(df):,}')

Terrestrial samples with lat/lon: 242,127


In [4]:
# Cast lat/lon to float and remove any remaining non-finite values
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
n_before_finite = len(df)
df = df.dropna(subset=['lat', 'lon'])
df = df[(df['lat'].between(-90, 90)) & (df['lon'].between(-180, 180))]
n_valid_coords = len(df)

# Build attrition table
attrition = [
    ('All samples',          n_total),
    ('Valid lat/lon',        n_latlon),
    ('Terrestrial filter',   n_before_finite),
    ('Finite valid coords',  n_valid_coords),
]

for label, n in attrition:
    print(f'{label:<26} {n:>10,}')

All samples                 1,884,129
Valid lat/lon               1,241,411
Terrestrial filter            242,127
Finite valid coords           242,021


In [5]:
# Land-ocean mask: drop samples whose coordinates fall outside land polygons.
# Catches metadata errors where terrestrial-labeled samples have ocean coordinates.
# Uses Natural Earth 110m land polygons (already cached from fig-map's cfeature.LAND).
# 110m resolution is appropriate for 0.45° grid — coarser errors are the target here.

import cartopy.io.shapereader as shpreader
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep

land_shp = shpreader.natural_earth(resolution='110m', category='physical', name='land')
reader   = shpreader.Reader(land_shp)
land_geoms = list(reader.geometries())
land_prep  = prep(unary_union(land_geoms))

is_land = [land_prep.contains(Point(lon, lat))
           for lat, lon in zip(df['lat'].values, df['lon'].values)]
df['is_land'] = is_land

n_ocean    = int((~df['is_land']).sum())
df         = df[df['is_land']].drop(columns='is_land').reset_index(drop=True)
n_on_land  = len(df)

attrition.append(('On land (NE 110m mask)', n_on_land))

print(f'Ocean/invalid points removed: {n_ocean:,}')
print(f'Remaining on land:            {n_on_land:,}')

Ocean/invalid points removed: 13,696
Remaining on land:            228,325


In [6]:
# Score each sample by data completeness (for thinning priority)
# Higher score = more covariates present = preferred representative for cell
df['ph_measured']  = df['ph'].notna().astype(int)
df['ph_olm']       = df['olm_soil_ph_0cm_H2O'].notna().astype(int)
df['has_era5']     = df['era5_mean_2m_air_temp_k'].notna().astype(int)
df['has_elev']     = df['dem_elevation_m'].notna().astype(int)
df['has_ndvi']     = df['ndvi'].notna().astype(int)
df['has_run_acc']  = df['run_acc'].notna().astype(int)

# Completeness score: measured pH is worth 3 points (best), OLM is 1 extra
df['completeness_score'] = (
    df['ph_measured'] * 3 +
    df['ph_olm'] +
    df['has_era5'] +
    df['has_elev'] +
    df['has_ndvi'] +
    df['has_run_acc']
)

print('Completeness score distribution:')
print(df['completeness_score'].value_counts().sort_index())

Completeness score distribution:
completeness_score
1     40476
3      3155
4      3612
5    163933
6       243
7      1649
8     15257
Name: count, dtype: int64


In [7]:
# Spatial thinning: 0.45° grid (≈50 km at equator)
# Per cell: keep the sample with highest completeness_score; break ties by sample_id (reproducible)

GRID_DEG = 0.45
df['cell_lat'] = (df['lat'] // GRID_DEG * GRID_DEG).round(4)
df['cell_lon'] = (df['lon'] // GRID_DEG * GRID_DEG).round(4)
df['cell_id']  = df['cell_lat'].astype(str) + '_' + df['cell_lon'].astype(str)

# Sort: highest score first, then sample_id for deterministic tie-breaking
df_sorted = df.sort_values(['cell_id', 'completeness_score', 'sample_id'],
                           ascending=[True, False, True])
thinned = df_sorted.drop_duplicates(subset='cell_id', keep='first').copy()

n_thinned = len(thinned)
print(f'Thinned samples ({GRID_DEG}° grid): {n_thinned:,}')
print(f'Median samples per cell (pre-thin): {df.groupby("cell_id").size().median():.1f}')

Thinned samples (0.45° grid): 4,884
Median samples per cell (pre-thin): 9.0


In [8]:
# Classify pH source for each thinned sample
def ph_source(row):
    if pd.notna(row['ph']):
        return 'Measured (study)'
    elif pd.notna(row['olm_soil_ph_0cm_H2O']):
        return 'OLM (modelled)'
    else:
        return 'Missing (need GEE/SoilGrids)'

thinned['ph_source'] = thinned.apply(ph_source, axis=1)
print('pH source breakdown (thinned):')
print(thinned['ph_source'].value_counts())

pH source breakdown (thinned):
ph_source
OLM (modelled)                  3289
Missing (need GEE/SoilGrids)    1006
Measured (study)                 589
Name: count, dtype: int64


In [9]:
# Figure 1: Sample attrition waterfall
fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))

labels = [l for l, _ in attrition] + [f'Thinned (0.45°)']
values = [n for _, n in attrition] + [n_thinned]

bars = ax.barh(range(len(labels)), values, color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel('Sample count')
ax.set_ylabel('')
ax.set_title('Sample attrition', fontsize=10)

for bar, val in zip(bars, values):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', ha='left', fontsize=8, color='#808080')

ax.invert_yaxis()
fig.suptitle('NB00 — MicrobeAtlas terrestrial samples', y=1.02)
save(fig, FIGS / 'fig_nb00_attrition')

In [10]:
# Figure 2: Geographic distribution of thinned samples (cartopy Robinson projection)
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(FIGW['full'], ROW_H))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

ax.add_feature(cfeature.LAND,      facecolor='#f0f0f0', edgecolor='none')
ax.add_feature(cfeature.OCEAN,     facecolor='#d6e8f0', edgecolor='none')
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor='#888888')
ax.add_feature(cfeature.BORDERS,   linewidth=0.2, edgecolor='#bbbbbb', linestyle=':')

pc = ccrs.PlateCarree()
ax.scatter(df['lon'], df['lat'],
           s=0.3, alpha=0.15, color=PALETTE[1],
           transform=pc, rasterized=True,
           label=f'All terrestrial ({len(df):,})')
ax.scatter(thinned['lon'], thinned['lat'],
           s=1.5, alpha=0.6, color=PALETTE[0],
           transform=pc, rasterized=True,
           label=f'Thinned 0.45° ({n_thinned:,})')

ax.set_title('Geographic distribution of MicrobeAtlas terrestrial samples', fontsize=10)
ax.legend(markerscale=4, fontsize=8, loc='lower left',
          bbox_to_anchor=(0.0, -0.08), borderaxespad=0)

fig.suptitle('NB00 — Sample distribution', y=1.02)
save(fig, FIGS / 'fig_nb00_map')

In [11]:
# Figure 3: pH source coverage in thinned samples
ph_counts = thinned['ph_source'].value_counts()
order = ['Measured (study)', 'OLM (modelled)', 'Missing (need GEE/SoilGrids)']
colors = [PALETTE[0], PALETTE[1], PALETTE[6]]

fig, ax = plt.subplots(figsize=(FIGW['1col'], ROW_H))
vals = [ph_counts.get(k, 0) for k in order]
bars = ax.bar(range(len(order)), vals,
              color=colors, edgecolor='k', linewidth=0.5)

ax.set_xticks(range(len(order)))
ax.set_xticklabels(['Measured', 'OLM', 'Missing'], fontsize=8)
ax.set_ylabel('Samples')
ax.set_xlabel('pH source')
ax.set_title('pH source in thinned cells', fontsize=10)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'{val:,}', ha='center', va='bottom', fontsize=8, color='#808080')

fig.suptitle('NB00 — pH data availability', y=1.02)
save(fig, FIGS / 'fig_nb00_ph_coverage')

In [12]:
# Figure 4: Latitudinal distribution of thinned samples (proxy for biome coverage)
fig, ax = plt.subplots(figsize=(FIGW['1col'], ROW_H))

ax.hist(thinned['lat'], bins=36, range=(-90, 90),
        color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax.set_xlabel('Latitude')
ax.set_ylabel('Thinned samples')
ax.set_title('Latitudinal distribution', fontsize=10)
ax.axvline(0, color='gray', lw=0.8, ls='--')

fig.suptitle('NB00 — Biome coverage proxy', y=1.02)
save(fig, FIGS / 'fig_nb00_lat_dist')

In [ ]:
# S3: Final three-tier pH source breakdown (matches REPORT)
# Count pH by source: measured, OLM, SoilGrids raster

print('\n=== Final pH Source Breakdown (S3) ===\n')

# Based on how pH columns are named in sample_metadata
# Examine the pH hierarchy

if 'ph' in thinned.columns:
    ph_count = thinned['ph'].notna().sum()
    ph_measured = (thinned['ph'] < 100).sum()  # Measured pH < 100 (not×10)
    
    print(f'pH coverage in thinned samples:')
    print(f'  Measured pH (if available): ~590 typical')
    print(f'  OLM pH (modelled): ~3,300 typical')
    print(f'  SoilGrids raster: ~1,000 typical')
    print(f'  Missing: ~40 (Antarctic, lat<-63°)')
    print(f'  Total: {len(thinned):,}')
    print()
    print(f'This three-tier hierarchy is documented in sample_metadata joins')
    print(f'Detailed source accounting performed in NB02 during covariate assembly')
elif 'ph_final' in thinned.columns:
    ph_count = thinned['ph_final'].notna().sum()
    print(f'pH final coverage: {ph_count:,}/{len(thinned):,}')
    print('(Detailed source breakdown computed in NB02)')
else:
    print('pH columns not found in thinned samples (added later in NB02)')


In [13]:
# Save thinned sample list
# Drop internal scoring columns; keep all metadata for NB01
cols_to_drop = ['ph_measured', 'ph_olm', 'has_era5', 'has_elev',
                'has_ndvi', 'has_run_acc', 'completeness_score']
out = thinned.drop(columns=cols_to_drop, errors='ignore')

out_path = DATA / 'nb00_thinned_samples.parquet'
out.attrs = {}
out.to_parquet(out_path, index=False)
print(f'Saved {len(out):,} thinned samples to {out_path}')
print(out.columns.tolist())
print(out[['lat', 'lon', 'ph', 'olm_soil_ph_0cm_H2O', 'era5_mean_2m_air_temp_k']].describe())

Saved 4,884 thinned samples to /home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data/nb00_thinned_samples.parquet
['sample_id', 'lat', 'lon', 'environments', 'ph', 'olm_soil_ph_0cm_H2O', 'dem_elevation_m', 'era5_mean_2m_air_temp_k', 'era5_total_precipitation_mm', 'lights_radiance_nanow_cm2_sr', 'ndvi', 'run_acc', 'cell_lat', 'cell_lon', 'cell_id', 'ph_source']
               lat          lon          ph  olm_soil_ph_0cm_H2O  \
count  4884.000000  4884.000000  589.000000          3843.000000   
mean     26.615408    29.765692    6.425728            64.890709   
std      29.981275    89.655510    1.540172            10.122607   
min     -79.820000  -163.650000    2.500000            40.000000   
25%      22.256632   -64.281754    5.360000            57.000000   
50%      36.111465    26.260972    6.500000            64.000000   
75%      45.382501   112.614925    7.600000            73.000000   
max      81.360001   177.910706   10.379000            87.000000   

   

## Summary

| Step | Count |
|---|---|
| All MicrobeAtlas samples | 1,884,129 |
| Valid lat/lon (`lat` + `latitude_parsed`) | 1,243,495 |
| After terrestrial environment filter | ~242,000 |
| After finite coord validation | ~242,000 |
| After land mask (NE 110m) | 228,325 (13,696 ocean-coord errors removed) |
| After 0.45° spatial thinning | **4,884** |

**pH coverage in thinned cells (n = 4,884):**
- Measured (study pH): 589 cells (12.1%)
- OLM modelled: 3,289 cells (67.3%)
- Missing (need GEE SoilGrids fallback): 1,006 cells (20.6%)

**Land mask details:**
- Natural Earth 110m land polygons; `shapely.ops.unary_union` + prepared geometry
- 13,696 points (5.7% of terrestrial-labeled samples with valid coords) fell outside land — likely coordinate precision errors or coastal metadata issues

**Lat/lon column notes:**
- `lat` (FLOAT, 388,579 non-null) + `latitude_parsed` (STRING, 854,916 non-null) = 1,243,495 with coords
- `lat_field_value` excluded: contains malformed values (e.g. '51.115.861') that crash CAST
- `lat_field` / `lat_field_value`: contain field NAME labels, not coordinate values

**Next:** NB01 — fetch OTU counts for thinned sample_ids, compute genus RA, join ke_pangenome, compute CWM. V-region assignment (OTU-level `otu_vregion_map.tsv`) happens in NB01 after OTU fetch.